# Stacked Ensemble v1 (OOF, Brier-Optimized)

This notebook trains and evaluates the new stack pipeline:
- Elo v2
- Women Power Ratings (in-house)
- LogReg v2 + interactions
- XGBoost (+ CatBoost fallback)
- Meta logistic stacker


In [1]:
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_men_data, load_women_data
from src.feature_engineering import (
    parse_seeds, compute_season_stats, compute_massey_features,
    compute_conference_strength, compute_efficiency, build_team_features,
    build_matchup_matrix
)
from src.model_elo_v2 import EloConfig, compute_elo_ratings_v2, build_elo_lookup
from src.model_logreg_v2 import LogRegV2Config
from src.model_stack_v1 import StackConfig, train_stack_final, evaluate_stack_holdout
from src.submission_stack_v1 import generate_submission_stacked
from src.women_rankings_v1 import WomenRankConfig, compute_women_power_ratings, merge_women_rank_features


In [2]:
data_dir = f'{project_root}/data'
m_data = load_men_data(data_dir)
w_data = load_women_data(data_dir)

# Men team features
m_elo = compute_elo_ratings_v2(m_data['MComSsn'], EloConfig())
m_features = build_team_features(
    m_elo,
    seeds_df=parse_seeds(m_data['MTrnySeeds']),
    stats_df=compute_season_stats(m_data['MDetSsn']),
    massey_df=compute_massey_features(m_data['MOrdinals']),
    conf_df=compute_conference_strength(m_data['MConf'], m_elo),
    efficiency_df=compute_efficiency(m_data['MDetSsn']),
)

# Women team features + in-house women rankings
w_elo = compute_elo_ratings_v2(w_data['WComSsn'], EloConfig())
w_base_features = build_team_features(
    w_elo,
    seeds_df=parse_seeds(w_data['WTrnySeeds']),
    stats_df=compute_season_stats(w_data['WDetSsn']),
    conf_df=compute_conference_strength(w_data['WConf'], w_elo),
    efficiency_df=compute_efficiency(w_data['WDetSsn']),
)
wpr = compute_women_power_ratings(w_data['WComSsn'], WomenRankConfig())
w_features = merge_women_rank_features(w_base_features, wpr)

m_matchups = build_matchup_matrix(m_data['MDetTrny'], m_features)
w_matchups = build_matchup_matrix(w_data['WDetTrny'], w_features)


In [ ]:
men_lr_cfg = LogRegV2Config(
    base_features=['Elo_diff', 'SeedNum_diff', 'Rank_POM_diff', 'Off_Eff_diff', 'Win_pct_diff'],
    interaction_pairs=[('Elo_diff', 'SeedNum_diff'), ('Elo_diff', 'Rank_POM_diff'), ('Off_Eff_diff', 'Win_pct_diff')],
    C=0.2,
)
women_lr_cfg = LogRegV2Config(
    base_features=['Elo_diff', 'SeedNum_diff', 'Net_Eff_diff', 'PPG_diff', 'PPG_allowed_diff', 'WPR_Rating_diff', 'WPR_SOS_diff'],
    interaction_pairs=[('Elo_diff', 'SeedNum_diff'), ('Elo_diff', 'Net_Eff_diff'), ('PPG_diff', 'PPG_allowed_diff')],
    C=1.0,
)

men_features_cfg = {
    'xgb_features': ['Elo_diff', 'SeedNum_diff', 'Rank_POM_diff', 'Off_Eff_diff', 'Win_pct_diff', 'Net_Eff_diff'],
    'cb_features': ['Elo_diff', 'SeedNum_diff', 'Rank_POM_diff', 'Off_Eff_diff', 'Win_pct_diff'],
}
women_features_cfg = {
    'xgb_features': ['Elo_diff', 'SeedNum_diff', 'Net_Eff_diff', 'PPG_diff', 'PPG_allowed_diff', 'WPR_Rating_diff', 'WPR_SOS_diff'],
    'cb_features': ['Elo_diff', 'SeedNum_diff', 'Net_Eff_diff', 'PPG_diff', 'WPR_Rating_diff'],
}

holdout_cutoff = 2022
holdout_max_train_season = holdout_cutoff - 1

print(f'True holdout protocol: train <= {holdout_max_train_season}, evaluate >= {holdout_cutoff}')

# 1) Train on pre-holdout seasons only for unbiased holdout evaluation.
holdout_stack_cfg = StackConfig(
    train_cutoff=holdout_cutoff,
    oof_start_season=2010,
    oof_end_season=holdout_max_train_season,
)

men_artifacts_holdout = train_stack_final(
    m_matchups, men_features_cfg, men_lr_cfg, holdout_stack_cfg, max_train_season=holdout_max_train_season
)
women_artifacts_holdout = train_stack_final(
    w_matchups, women_features_cfg, women_lr_cfg, holdout_stack_cfg, max_train_season=holdout_max_train_season
)

print('Men LOSO OOF (train-only):', men_artifacts_holdout['oof_summary'])
print('Women LOSO OOF (train-only):', women_artifacts_holdout['oof_summary'])

men_holdout = evaluate_stack_holdout(m_matchups, men_artifacts_holdout, train_cutoff=holdout_cutoff)
women_holdout = evaluate_stack_holdout(w_matchups, women_artifacts_holdout, train_cutoff=holdout_cutoff)

print('Men holdout:', men_holdout)
print('Women holdout:', women_holdout)
combined_holdout = (
    men_holdout['holdout_brier'] * men_holdout['n_holdout']
    + women_holdout['holdout_brier'] * women_holdout['n_holdout']
) / (men_holdout['n_holdout'] + women_holdout['n_holdout'])
print(f'Combined holdout: {combined_holdout:.5f}')

# 2) Retrain on all seasons for final submission generation.
final_stack_cfg = StackConfig(train_cutoff=holdout_cutoff, oof_start_season=2010, oof_end_season=2025)
men_artifacts = train_stack_final(m_matchups, men_features_cfg, men_lr_cfg, final_stack_cfg, max_train_season=2025)
women_artifacts = train_stack_final(w_matchups, women_features_cfg, women_lr_cfg, final_stack_cfg, max_train_season=2025)


In [ ]:
elo_lookup = build_elo_lookup(pd.concat([m_elo, w_elo], ignore_index=True))
sub = generate_submission_stacked(
    f'{project_root}/data/SampleSubmissionStage1.csv',
    men_artifacts,
    women_artifacts,
    m_features,
    w_features,
    elo_lookup,
)
sub.to_csv(f'{project_root}/submissions/stage1_stack_v1.csv', index=False)
print('Saved submissions/stage1_stack_v1.csv')
